*Importando o Pandas*

In [1]:
import pandas as pd
import numpy as np

#definindo quantidade de linhas e colunas visiveis
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)

*Importação de arquivos EXCEL*

In [2]:
caminho = ('G:/Drives compartilhados/Qualidade/ACURÁCIA/DISCREPÂNCIA/DISCREPÂNCIA DIAGNÓSTICA/Discrepância 2026/Mar/')
df = pd.read_excel(caminho + 'Discrepancia Osasco Mar 2026.xlsx')
df_bck = df.copy()

In [ ]:
df['Ds setor atendimento'].value_counts()

*Criando o dataframe de CIDs.*

In [6]:
df['Cor'] = ''

In [ ]:
df.loc[df['Validacao cid'] == "CID DIFERENTE", 'Cor'] = ""

*Remover linhas que não contem resposta*

In [8]:
# O raciocínio: Localize as linhas onde isnull() é True e mude a coluna 'Cor'
df.loc[df['Diagnóstico pós operatório'].isna() | df['Diagnóstico pós operatório'].str.contains(r'^\s*$', regex=True), 'Cor'] = "Amarelo"
print(df['Diagnóstico pós operatório'].isna() | df['Diagnóstico pós operatório'].str.contains(r'^\s*$', regex=True))

0       False
1       False
2        True
3        True
4        True
        ...  
1812    False
1813     True
1814    False
1815     True
1816    False
Name: Diagnóstico pós operatório, Length: 1817, dtype: bool


*Removendo CIDs*

In [10]:
import numpy as np

# 1. Manter a limpeza de nulos inicial
df['Diagnostico pré operatório'] = df['Diagnostico pré operatório'].fillna('')

# 2. Identificação e Atribuição da Cor
padrao_cid = r"[A-Z]\d+(?:\.\d+)?"
df['Cor'] = np.where(
    df['Diagnostico pré operatório'].str.contains(padrao_cid, regex=True, na=False), 
    'Roxo', 
    ''
)

# 3. A "Lanterna" (Máscara Booleana)
# Criamos um filtro para identificar onde a Cor está vazia
mask_vazio = df['Cor'] == ''

# 4. Aplicação Cirúrgica
# Aplicamos as transformações de string APENAS nas linhas da máscara
df.loc[mask_vazio, 'Diagnostico pré operatório'] = (
    df.loc[mask_vazio, 'Diagnostico pré operatório']
    .str.replace("_x000D_", " ", regex=False)
    .str.strip()
)

print(df[['Diagnostico pré operatório', 'Cor']].head())

  Diagnostico pré operatório Cor
0                        TPP    
1            PARTO EXPULSIVO    
2                               
3                               
4                               


In [13]:
import numpy as np

# 1. Manter a limpeza de nulos inicial
df['Diagnóstico pós operatório'] = df['Diagnóstico pós operatório'].fillna('')

# 2. Identificação e Atribuição da Cor
padrao_cid = r"[A-Z]\d+(?:\.\d+)?"
df['Cor'] = np.where(
    df['Diagnóstico pós operatório'].str.contains(padrao_cid, regex=True, na=False), 
    'Roxo', 
    ''
)

# 3. A "Lanterna" (Máscara Booleana)
# Criamos um filtro para identificar onde a Cor está vazia
mask_vazio = df['Cor'] == ''

# 4. Aplicação Cirúrgica
# Aplicamos as transformações de string APENAS nas linhas da máscara
df.loc[mask_vazio, 'Diagnóstico pós operatório'] = (
    df.loc[mask_vazio, 'Diagnóstico pós operatório']
    .str.replace("_x000D_", " ", regex=False)
    .str.strip()
)

print(df[['Diagnóstico pós operatório', 'Cor']].head())

  Diagnóstico pós operatório Cor
0                        TPP    
1                    O MESMO    
2                               
3                               
4                               


In [14]:
df.loc[df['Diagnostico pré operatório'] == df['Diagnóstico pós operatório'],'Cor'] = 'Verde'

*Classificação de Dados*

In [ ]:
df.sort_values(by=["Nome paciente"])

In [ ]:
contagem = df['Ds setor atendimento'].value_counts()
print(contagem)

*Filtrando os dados*

In [15]:
# Filtrando linhas coma palavra Convalescenca
conva = df['Diagnóstico pós operatório'].str.contains('CONVALESCENCA', na=False)
df.loc[conva, 'Cor'] = "Amarelo"
# Exibindo as linhas corretas
print(df[df['Diagnóstico pós operatório'].str.contains('CONVALESCENCA', na=False)])

Empty DataFrame
Columns: [þÿData, Nome paciente, Atendimento, Nome do Medico, Cirurgia agendada, Cirurgia realizada, Diagnostico pré operatório, Diagnóstico pós operatório, Cid, Cor]
Index: []


In [16]:
outros = df['Diagnóstico pós operatório'].str.contains('OUTROS', na=False)
df.loc[outros, 'Cor'] = " Amarelo"

print(df[df['Diagnóstico pós operatório'].str.contains('OUTROS', na=False)])

Empty DataFrame
Columns: [þÿData, Nome paciente, Atendimento, Nome do Medico, Cirurgia agendada, Cirurgia realizada, Diagnostico pré operatório, Diagnóstico pós operatório, Cid, Cor]
Index: []


In [17]:
segui = df['Diagnóstico pós operatório'].str.contains('Seguimentos', na=False)
df.loc[segui, 'Cor'] = " Amarelo"

print(df[df['Diagnóstico pós operatório'].str.contains('Seguimentos', na=False)])

Empty DataFrame
Columns: [þÿData, Nome paciente, Atendimento, Nome do Medico, Cirurgia agendada, Cirurgia realizada, Diagnostico pré operatório, Diagnóstico pós operatório, Cid, Cor]
Index: []


In [24]:
palavras = df['Diagnóstico pós operatório'].str.contains(
    'Mesmo|Idem|NDN|PO|POI|POS|PÓS',
    na=False
)

df.loc[palavras, 'Cor'] = "Amarelo"

print(df.loc[palavras, ['Diagnóstico pós operatório', 'Cor']])

                             Diagnóstico pós operatório      Cor
16                                     POI DE CESARIANA  Amarelo
18                                  POI DE PARTO NORMAL  Amarelo
52     # PO (02/03/26) POSTECTOMIA- Sem intercorrências  Amarelo
117                                 POI DE PARTO NORMAL  Amarelo
131                                    POI DE CESARIANA  Amarelo
173   POI RESSECÇÃO SEGMENTAR MAMA ESQUERDA + RECONS...  Amarelo
239   ELAGASTROPARESIADIFICULADADE DE DGLUTIÇÃOHIPER...  Amarelo
266                                    POI DE CESARIANA  Amarelo
270   TROMBOSE HEMORROIDWERIAABSCESSO ANAL INTERESFI...  Amarelo
278   ABSCESSO ISQUIORRETAL EM FERRADURAFISTULA EM F...  Amarelo
288   TROMBOSE HEMORROIDARIAPROLAPSO MUCOSO EDEMACIA...  Amarelo
299   ABSCESSO ISQUIORRETAL E INTERESFINCTERIANOFIST...  Amarelo
331   POI OSTEOSSINTESE DE 5 MTC + REPARO LIGAMENTAR...  Amarelo
358                                                 NDN  Amarelo
362                      

*Verificando score entre palavras*

In [18]:
from thefuzz import process
from thefuzz import fuzz

df['Diagnóstico pós operatório'] = df['Diagnóstico pós operatório'].fillna('')

df['score_fuzz.ratio'] = 0
df['score_fuzz.ratio'] = df['score_fuzz.ratio'].astype('float')
df['score_fuzz.token_sort_ratio'] = df['score_fuzz.ratio']
df['score_fuzz.partial_ratio'] = df['score_fuzz.ratio']

for linha in df.index:
    
    texto1 = df.loc[linha, 'Diagnostico pré operatório']
    texto2 = df.loc[linha, 'Diagnóstico pós operatório']
    # ===================================================
    melhor, score = process.extractOne(
        query=texto1,
        choices=[texto2],
        scorer=fuzz.ratio
    )
    df.loc[linha, 'score_fuzz.ratio'] = score
    # ===================================================
    melhor, score = process.extractOne(
        query=texto1,
        choices=[texto2],
        scorer=fuzz.token_sort_ratio
    )
    df.loc[linha, 'score_fuzz.token_sort_ratio'] = score
    # ===================================================
    melhor, score = process.extractOne(
        query=texto1,
        choices=[texto2],
        scorer=fuzz.partial_ratio
    )
    df.loc[linha, 'score_fuzz.partial_ratio'] = score
    # ===================================================
    #display(df.loc[linha:linha, ['Diagnostico pre','Diagnostico pós','score']])

Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: '']
Applied processor reduces input query to empty string, all comparisons will have s

In [19]:
df_bck.loc[140, 'Diagnostico pré operatório'][2]

'D'

In [20]:
display(df_bck.loc[57:57])

display(df.loc[57:57])

,þÿData,Nome paciente,Atendimento,Nome do Medico,Cirurgia agendada,Cirurgia realizada,Diagnostico pré operatório,Diagnóstico pós operatório,Cid
57,2026-03-02,Joao Gomes de Almeida Filho,52819237,Rodrigo Mendonça Dionisio,Cateterismo de Câmaras Cardíacas Esquerdas E C...,Cateterismo de Câmaras Cardíacas Esquerdas E C...,,,


,þÿData,Nome paciente,Atendimento,Nome do Medico,Cirurgia agendada,Cirurgia realizada,Diagnostico pré operatório,Diagnóstico pós operatório,Cid,Cor,score_fuzz.ratio,score_fuzz.token_sort_ratio,score_fuzz.partial_ratio
57,2026-03-02,Joao Gomes de Almeida Filho,52819237,Rodrigo Mendonça Dionisio,Cateterismo de Câmaras Cardíacas Esquerdas E C...,Cateterismo de Câmaras Cardíacas Esquerdas E C...,,,,Verde,100.0,100.0,100.0


In [21]:
display(df[['Diagnostico pré operatório','Diagnóstico pós operatório','score_fuzz.ratio','score_fuzz.token_sort_ratio','score_fuzz.partial_ratio']].sample(20))

,Diagnostico pré operatório,Diagnóstico pós operatório,score_fuzz.ratio,score_fuzz.token_sort_ratio,score_fuzz.partial_ratio
768,,,100.0,100.0,100.0
1093,fistula dural,fistula dural pos trombose venosa cerebral,47.0,47.0,100.0
368,M532 Instabilidades da coluna vertebral.ESPOND...,M532 Instabilidades da coluna vertebral.ESPOND...,100.0,100.0,100.0
322,Prolapso de reto encarcerado com sangramento a...,Prolapso de reto encarcerado com sangramento a...,100.0,100.0,100.0
91,desvio septal,desvio septal,100.0,100.0,100.0
225,,,100.0,100.0,100.0
1485,VARIZES,TCVB,18.0,18.0,40.0
1393,TRABALHO DE PARTO,O MESMO,33.0,33.0,43.0
1657,,,100.0,100.0,100.0
926,OBESIDADE MÓRBIDA,O MESMO,33.0,26.0,55.0


*Retirar linhas com score acima de 70*

In [22]:
roxo = (
        (df['score_fuzz.ratio'] > 50) &
        (df['score_fuzz.token_sort_ratio'] > 50) &
        (df['score_fuzz.partial_ratio'] > 50))

condicao_vazio = (df['Cor'].isna()) | (df['Cor'] == '')

df.loc[roxo & condicao_vazio, 'Cor'] = 'Roxo'

In [23]:
df.to_excel("Discrepancia Osasco Mar 2026(teste1).xlsx")